In [14]:
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

from utils.helpers import *

In [15]:
input_path = 'data/'
output_path = 'data/preprocessed'

x_train, x_test, y_train, train_ids, test_ids = load_csv_data(data_path='data/')

In [16]:
def classify_columns(x):
    """
    Classifies columns into numerical, symbolic, cyclic, and binary 
    based purely on the number of unique values in the NumPy array.
    """
    cyclic_indices = [2, 4, 5]  # FMONTH, IMONTH, IDAY

    cyclic_cols = list(cyclic_indices)
    symbolic_cols = []  
    binary_cols = []
    numerical_cols = []

    num_features = x.shape[1]
    
    for col in range(num_features):
        if col in cyclic_cols:
            continue

        col_data = x[:, col]
        mask = (~np.isnan(col_data)) & (col_data != -999.0)
        valid_data = col_data[mask]
        
        if len(valid_data) == 0:
            continue
            
        nunique = len(np.unique(valid_data))
        
        if nunique <= 2:
            binary_cols.append(col)
        elif 2 < nunique < 25:
            symbolic_cols.append(col)
        else:
            numerical_cols.append(col)
            
    return numerical_cols, symbolic_cols, cyclic_cols, binary_cols


def drop_high_missing_columns(x_train, x_test, threshold=0.50, missing_val=-999.0):
    """Drops columns from x_train and x_test where missing ratio in x_train > threshold."""
    is_missing = np.isnan(x_train) | (x_train == missing_val)
    missing_ratios = np.mean(is_missing, axis=0)
    
    cols_to_keep = np.where(missing_ratios <= threshold)[0]
    
    print(f"Original columns: {x_train.shape[1]}")
    print(f"Dropped {x_train.shape[1] - len(cols_to_keep)} columns with > {int(threshold * 100)}% missing values.")
    print(f"Remaining columns: {len(cols_to_keep)}")
    
    return x_train[:, cols_to_keep], x_test[:, cols_to_keep]

# 1. Drop high-missingness columns first
x_train_clean, x_test_clean = drop_high_missing_columns(x_train, x_test, threshold=0.50)

# 2. Classify remaining columns on the cleaned matrix
numerical_cols, symbolic_cols, cyclic_cols, binary_cols = classify_columns(x_train_clean)


Original columns: 321
Dropped 147 columns with > 50% missing values.
Remaining columns: 174


# 1. Variable Encoding

In [17]:
def fit_transform_encodings(
    x_train, x_test, num_cols, sym_cols, cyc_cols, bin_cols
):
    """Simplifies encoding:

    - Binary features -> standardized to {0, 1}
    - Symbolic features -> mapped to ordinal integers (0, 1, 2, ...)
    - Cyclic features -> Sine & Cosine transformation
    - Numerical features -> preserved as continuous values

    Keeps column dimension at ~177 and eliminates multicollinearity.
    """
    x_tr = np.copy(x_train)
    x_te = np.copy(x_test)

    # 1. Binary Encoding (Map to 0/1)
    for col in bin_cols:
        col_tr = x_tr[:, col]
        mask_tr = (~np.isnan(col_tr)) & (col_tr != -999.0)
        vals = np.unique(col_tr[mask_tr])

        if len(vals) >= 1:
            val0 = vals[0]
            val1 = vals[1] if len(vals) > 1 else None

            x_tr[x_tr[:, col] == val0, col] = 0
            x_te[x_te[:, col] == val0, col] = 0
            if val1 is not None:
                x_tr[x_tr[:, col] == val1, col] = 1
                x_te[x_te[:, col] == val1, col] = 1

    # 2. Symbolic Encoding (Ordinal Integer Mapping: 0, 1, 2...)
    for col in sym_cols:
        col_tr = x_tr[:, col]
        mask_tr = (~np.isnan(col_tr)) & (col_tr != -999.0)
        vals = np.unique(col_tr[mask_tr])

        for i, val in enumerate(vals):
            x_tr[x_tr[:, col] == val, col] = i
            x_te[x_te[:, col] == val, col] = i

    # 3. Cyclic Encoding (Replace 3 cyclic columns with sin/cos pairs)
    cols_to_keep = [i for i in range(x_tr.shape[1]) if i not in cyc_cols]
    blocks_tr = [x_tr[:, cols_to_keep]]
    blocks_te = [x_te[:, cols_to_keep]]

    for col in cyc_cols:
        col_tr = x_tr[:, col]
        mask_tr = (~np.isnan(col_tr)) & (col_tr != -999.0)
        max_val = np.max(col_tr[mask_tr]) if np.any(mask_tr) else 1.0

        sin_tr = np.sin(2 * np.pi * col_tr / max_val).reshape(-1, 1)
        cos_tr = np.cos(2 * np.pi * col_tr / max_val).reshape(-1, 1)
        sin_tr[~mask_tr] = np.nan
        cos_tr[~mask_tr] = np.nan
        blocks_tr.extend([sin_tr, cos_tr])

        col_te = x_te[:, col]
        mask_te = (~np.isnan(col_te)) & (col_te != -999.0)
        sin_te = np.sin(2 * np.pi * col_te / max_val).reshape(-1, 1)
        cos_te = np.cos(2 * np.pi * col_te / max_val).reshape(-1, 1)
        sin_te[~mask_te] = np.nan
        cos_te[~mask_te] = np.nan
        blocks_te.extend([sin_te, cos_te])

    x_train_out = np.concatenate(blocks_tr, axis=1)
    x_test_out = np.concatenate(blocks_te, axis=1)

    return x_train_out, x_test_out

In [18]:
# Transform train and test matrices together using Fit-Transform pattern
x_train_encoded, x_test_encoded = fit_transform_encodings(
    x_train_clean, x_test_clean, numerical_cols, symbolic_cols, cyclic_cols, binary_cols
)

print("Original x_train shape:", x_train.shape)
print("Encoded x_train shape: ", x_train_encoded.shape)
print("Encoded x_test shape:  ", x_test_encoded.shape)

Original x_train shape: (328135, 321)
Encoded x_train shape:  (328135, 177)
Encoded x_test shape:   (109379, 177)


# 2. Missing Values Imputation

In [19]:
def impute_missing_values(x_train_enc, x_test_enc, missing_val=-999.0):
    """Imputes missing values across all columns using column medians."""
    x_tr = np.copy(x_train_enc)
    x_te = np.copy(x_test_enc)

    x_tr[x_tr == missing_val] = np.nan
    x_te[x_te == missing_val] = np.nan

    for col in range(x_tr.shape[1]):
        col_tr = x_tr[:, col]   
        mask_tr = ~np.isnan(col_tr)

        fill_val = np.nanmedian(col_tr) if np.any(mask_tr) else 0.0

        x_tr[np.isnan(x_tr[:, col]), col] = fill_val
        x_te[np.isnan(x_te[:, col]), col] = fill_val

    return x_tr, x_te


x_train_imp, x_test_imp = impute_missing_values(
    x_train_encoded, x_test_encoded
)

# 3. Outliers

In [20]:
def handle_outliers(x_train, x_test):
    """Clips extreme outliers across columns using Interquartile Range (IQR) bounds."""
    x_tr = np.copy(x_train)
    x_te = np.copy(x_test)
    
    for col in range(x_tr.shape[1]):
        q1 = np.percentile(x_tr[:, col], 25)
        q3 = np.percentile(x_tr[:, col], 75)
        iqr = q3 - q1
        
        if iqr > 0:
            lower_bound = q1 - 1.5 * iqr
            upper_bound = q3 + 1.5 * iqr
            x_tr[:, col] = np.clip(x_tr[:, col], lower_bound, upper_bound)
            x_te[:, col] = np.clip(x_te[:, col], lower_bound, upper_bound)
            
    return x_tr, x_te

x_train_no_out, x_test_no_out = handle_outliers(x_train_imp, x_test_imp)

# 4. Scaling

In [21]:
def standardize(x_train, x_test):
    """Standardizes features using Z-score normalization."""
    x_tr = np.copy(x_train)
    x_te = np.copy(x_test)
    
    mean = np.mean(x_tr, axis=0)
    std = np.std(x_tr, axis=0)
    
    # Prevent division by zero for constant columns
    std[std == 0] = 1.0
    
    x_tr_scaled = (x_tr - mean) / std
    x_te_scaled = (x_te - mean) / std
    
    return x_tr_scaled, x_te_scaled

x_train_scaled, x_test_scaled = standardize(x_train_no_out, x_test_no_out)

# 5. Data Balancing

In [22]:
# def balance_classes(x, y):
#     """Undersamples the majority class to balance the dataset."""
#     classes, counts = np.unique(y, return_counts=True)
#     if len(classes) < 2:
#         return x, y
        
#     majority_class = classes[np.argmax(counts)]
#     minority_class = classes[np.argmin(counts)]
    
#     idx_maj = np.where(y == majority_class)[0]
#     idx_min = np.where(y == minority_class)[0]
    
#     np.random.seed(42)
#     idx_maj_sampled = np.random.choice(idx_maj, size=len(idx_min), replace=False)
    
#     balanced_idx = np.concatenate([idx_min, idx_maj_sampled])
#     np.random.shuffle(balanced_idx)
    
#     return x[balanced_idx], y[balanced_idx]

# x_train_bal, y_train_bal = balance_classes(x_train_scaled, y_train)

In [23]:
def remove_collinear_and_constant_columns(x_train, x_test, tol=1e-8):
    """
    1. Removes constant columns (zero variance / all zeros).
    2. Removes duplicate columns.
    """
    x_tr = np.copy(x_train)
    x_te = np.copy(x_test)
    
    # 1. Remove constant columns
    variances = np.var(x_tr, axis=0)
    non_constant = variances > tol
    
    x_tr = x_tr[:, non_constant]
    x_te = x_te[:, non_constant]
    
    # 2. Remove duplicate columns
    _, unique_idx = np.unique(x_tr.T, axis=0, return_index=True)
    unique_idx = np.sort(unique_idx) # preserve original feature order
    
    x_tr_clean = x_tr[:, unique_idx]
    x_te_clean = x_te[:, unique_idx]
    
    print(f"Features reduced from {x_train.shape[1]} to {x_tr_clean.shape[1]} (removed constant/duplicate features).")
    return x_tr_clean, x_te_clean

# Clean x_train_scaled and x_test_scaled before saving
x_train_final, x_test_final = remove_collinear_and_constant_columns(x_train_scaled, x_test_scaled)

Features reduced from 177 to 174 (removed constant/duplicate features).


# Data inspection and saving 

In [24]:
import os
os.makedirs(output_path, exist_ok=True)

# Inspect shapes
print("--- Final Matrix Shapes ---")
print("x_train:", x_train_final.shape)
print("x_test: ", x_test_final.shape)
print("NaNs present:", np.isnan(x_train_scaled).any())  # Should print False

np.save(os.path.join(output_path, 'x_train_preprocessed.npy'), x_train_final)
np.save(os.path.join(output_path, 'x_test_preprocessed.npy'), x_test_final)

print("Preprocessing complete.")

--- Final Matrix Shapes ---
x_train: (328135, 174)
x_test:  (109379, 174)
NaNs present: False
Preprocessing complete.
